# 05 — Reinforcement learning: SAC allocation policy

Train a Soft Actor-Critic policy that maps a feature vector to an **allocation
weight in [0, 1]** per name — a continuous action, not a discrete buy/sell.

Three design points worth understanding before running it:

**The reward is a differential Sortino, net of turnover.**
`R_t = a_t · ret_{t+1} − cost·|a_t − a_{t−1}|` feeds an online Sortino whose
per-step increment is the reward. Summing those increments approximates the
Sortino ratio of the whole path, so a policy maximizing per-step reward is
maximizing a downside-aware ratio rather than raw return — the distinction that
stops it learning "hold maximum size always". Note the friction is a function of
the *action*: a cost charged as a constant every step shifts every action's
reward identically and therefore cannot penalize turnover at all.

**`gamma` defaults to 0.** Discounting assumes the action influences the next
state. A price-taking book does not move the market, so the observed state
sequence is exogenous, and bootstrapping a value function over it adds estimator
variance without adding signal. With `gamma=0` the critic learns
`Q(s,a) = E[r|s,a]` and the actor maximizes `Q − α·log π` — soft actor-critic
applied to what this decision actually is, a contextual bandit.

**Inference is deterministic.** SAC optimizes a stochastic policy — a squashed
Gaussian whose log-std head supplies the entropy term the algorithm is named
for. At scoring time the mean action is used instead: sampling would make two
runs of one backtest disagree.

Standalone: no `portfolio_agent` import. Needs `torch`.

## Setup

In [ ]:
# Dependencies. Torch is only needed by the two learned strategies (04, 05).
# !pip install -q pandas numpy pyarrow matplotlib huggingface_hub torch

import sys, pathlib

# afa_lab.py sits next to this notebook. On Colab (or anywhere the file is
# missing) fetch it from the repo — that is the only network call that touches
# GitHub, and nothing else here imports the portfolio_agent package.
if not pathlib.Path("afa_lab.py").exists():
    import urllib.request
    URL = ("https://raw.githubusercontent.com/3dwag98/afa/main/"
           "notebooks/standalone/afa_lab.py")
    urllib.request.urlretrieve(URL, "afa_lab.py")
    print("fetched afa_lab.py")

sys.path.insert(0, ".")
import afa_lab as L

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("toolkit loaded | torch available:", L.TORCH_AVAILABLE)

In [ ]:
# NSE large caps. Any symbol absent from the dataset is skipped rather than
# failing the run, so this list does not have to be exactly right.
UNIVERSE = [
    "RELIANCE",
    "TCS",
    "HDFCBANK",
    "INFY",
    "ICICIBANK",
    "HINDUNILVR",
    "ITC",
    "SBIN",
    "BHARTIARTL",
    "KOTAKBANK",
    "LT",
    "AXISBANK",
    "ASIANPAINT",
    "MARUTI",
    "SUNPHARMA",
    "TITAN",
    "ULTRACEMCO",
    "WIPRO",
    "NESTLEIND",
    "BAJFINANCE",
    "TATAMOTORS",
    "TATASTEEL",
    "POWERGRID",
    "NTPC",
    "ONGC",
    "HCLTECH",
    "JSWSTEEL",
    "GRASIM",
    "CIPLA",
    "COALINDIA"
]

START_DATE = "2018-01-01"
END_DATE   = None          # None = up to the dataset's last session
CACHE      = "data_cache"  # downloaded parquet files land here and are reused

print(len(UNIVERSE), "symbols requested")

In [ ]:
# Ingestion. One small parquet per symbol is pulled from the Hub dataset
# `vishnun0027/indian-market-historical-ohlcv` (2,421 NSE/BSE equities) and
# cleaned. Downloading per symbol rather than snapshotting the repo means a
# 30-name universe fetches 30 small files instead of 283 MB.
#
# Cleaning, in order: back-adjust OHLC by adj_close/close so a split is not read
# as a 90% crash, coerce numerics, drop unparseable dates and missing closes
# (rather than forward-filling, so a gap stays visible), and drop duplicate
# sessions keeping the last.
#
# If the Hub is unreachable the toolkit falls back to a synthetic panel and says
# so loudly. Synthetic results describe the generator, not the market.

panel = L.load_panel(UNIVERSE, start_date=START_DATE, end_date=END_DATE,
                     cache_dir=CACHE)

close = L.align_close_matrix(panel)
print(f"{len(panel)} symbols | {close.index.min().date()} -> {close.index.max().date()}"
      f" | {len(close)} sessions")

In [ ]:
# Features are computed per symbol and left NaN until each window has filled.
# They are never back-filled: a back-filled indicator is a look-ahead, and it is
# invisible in every metric downstream.
feature_panel = L.build_feature_panel(panel)

sample = feature_panel[sorted(feature_panel)[0]]
print(f"{len(sample.columns)} features:", list(sample.columns))
display(sample.dropna().tail(3))

In [ ]:
# Simulation settings, shared by every notebook so the strategies are comparable.
#
# execution_lag=1 is the property that keeps this honest: a signal computed from
# day t's close is traded into day t+1's return. The engine refuses lag=0.
config = L.BacktestConfig(
    initial_capital=1_000_000.0,
    cost_bps=25.0,        # all-in round trip for Indian cash equities
    max_weight=0.10,
    rebalance_days=5,     # weekly; the main control over turnover
    max_gross=1.0,        # long-only, unlevered
    execution_lag=1,
)

benchmark = L.equal_weight_benchmark(close, config)
print("equal-weight buy & hold:",
      {k: round(v, 4) for k, v in benchmark.stats.items()
       if k in ("cagr", "sharpe", "max_drawdown")})

## Train

Experience is **re-collected every epoch** from the current policy. Training
against a buffer collected once means the actor only ever sees a randomly
initialized policy's decisions and never the consequences of its own.

The policy is trained on the first 70% of history only; everything after is held
out.

In [ ]:
sac = L.train_sac(
    feature_panel, close,
    epochs=40,
    batch_size=256,
    learning_rate=3e-4,
    hidden_dim=128,
    gamma=0.0,              # contextual bandit; see above
    tau=0.005,
    friction_cost=0.008,
    train_fraction=0.70,
    gradient_steps=150,
    device="auto",
    seed=42,
)

print(f"\ntrained on data up to {sac['split_date'].date()}")
L.plot_training_curve(sac["history"], ["critic_loss", "actor_loss", "alpha"],
                      title="SAC training")

## The learned policy

`threshold` is what turns a continuous allocation into a portfolio. Below it the
policy is not asking for a position, and keeping those weights would put a token
holding in every name in the universe.

In [ ]:
sac_scores = L.sac_scores(sac, feature_panel, close, threshold=0.60)

held = (sac_scores > 0).sum(axis=1)
print(f"names held: mean {held.mean():.1f} | flat on {(held == 0).mean():.1%} of sessions")

In [ ]:
# What the policy actually learned to output. A distribution piled against 0 or 1
# means the entropy term collapsed; a flat one near 0.5 means it never learned
# to discriminate.
raw = L.sac_scores(sac, feature_panel, close, threshold=0.0)
values = raw.to_numpy().ravel()
values = values[values > 0]

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].hist(values, bins=60, color=L.PALETTE[0])
axes[0].axvline(0.60, color=L.PALETTE[1], linestyle="--", label="threshold")
axes[0].set_title("Distribution of allocation weights"); axes[0].legend(fontsize=8)
axes[0].grid(**L.GRID)

axes[1].plot(raw.index, raw.mean(axis=1), color=L.PALETTE[2], linewidth=1.1)
axes[1].axvline(sac["split_date"], color="#444444", linestyle=":", label="train/test cut")
axes[1].set_title("Mean allocation through time"); axes[1].legend(fontsize=8)
axes[1].grid(**L.GRID)
plt.tight_layout(); plt.show()

## Backtest

Against equal-weight buy-and-hold of the same names — the honest comparison for a long-only stock picker. Beating cash is not the question.

In [ ]:
result = L.run_backtest(sac_scores, close, config)

comparison = L.compare_stats({"sac": result, "equal weight": benchmark})
display(comparison)

## Analysis

In [ ]:
L.plot_equity({"sac": result}, title="sac vs equal weight",
              benchmark=benchmark.returns)

In [ ]:
L.plot_return_profile(result, "sac")

In [ ]:
L.plot_exposure(result, "sac")

In [ ]:
L.plot_weight_heatmap(result, title="sac: allocation over time")

In [ ]:
# Out-of-sample only — the policy never saw this period.
cut = sac["split_date"]
oos_close = close[close.index > cut]
oos = L.run_backtest(sac_scores[sac_scores.index > cut], oos_close, config)
oos_benchmark = L.equal_weight_benchmark(oos_close, config)

display(L.compare_stats({"sac (out-of-sample)": oos, "equal weight": oos_benchmark}))
L.plot_equity({"sac (out-of-sample)": oos}, benchmark=oos_benchmark.returns,
              title=f"SAC out-of-sample (from {cut.date()})", log_scale=False)

## A caveat this design carries openly

The reward is net of turnover, which depends on the previous allocation — but
the state does not carry that previous allocation. The policy is therefore
mildly **partially observed**: it is penalized for turnover it cannot see. That
is a deliberate trade to keep the state vector purely feature-based; adding
previous allocation as a twelfth input would fix it at the cost of a state
representation that inference has to track.

Reinforcement learning on a few thousand transitions of one universe is also
close to the smallest problem this method is worth using on. Treat the mechanism
as the deliverable, not the equity curve.

---

## What this does and does not show

Read before quoting any number above.

- **Survivorship.** The universe is today's large caps, applied to history. Names
  that were large caps in 2018 and are not now are absent, and they are absent
  precisely because they did badly. Every long-only result here is biased upward
  by an amount this notebook cannot measure. A point-in-time constituent list is
  the only fix, and this dataset does not carry one.
- **One universe, one period.** Thirty names over a few years is a single draw.
  The difference between two strategies here is well within what the draw alone
  could produce.
- **Costs are a flat 25 bps.** Real cost scales with size and with how illiquid
  the name is, and the fill is assumed at the close. A strategy whose edge is
  this side of costs is not distinguishable from one that has no edge.
- **No point-in-time fundamentals, no corporate actions beyond the price
  adjustment**, and no circuit-limit modelling. On Indian equities a
  circuit-locked session is untradeable, and the simulation will happily trade it.
- **Parameters were chosen, not fitted.** Nothing here is tuned on a held-out
  period. That is deliberate — tuning on this sample and reporting the result
  would be reporting the tuning.

The purpose of these notebooks is to make the mechanism legible and modifiable,
not to establish that any of these strategies makes money.